# GovernanceFund — 투표 → 비중 결정 알고리즘

**목적**: 투표 입력 → 포지션 비중(방향 + 레버리지) 산출 과정을 검증  
**구조**: Kahoot 스타일 UI 시뮬레이션 → 4가지 알고리즘 비교 → 공격적/보수적 프로파일

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ 라이브러리 로드 완료')

## 1. 펀드 프로파일 설정

**공격적**: 빠른 반응, 큰 조정폭  
**보수적**: 느린 반응, 작은 조정폭

In [ ]:
FUND_PROFILES = {
    'aggressive': {
        'MAX_STEP': 25,      # 한 투표 라운드에 최대 25%p 조정
        'ALPHA': 0.6,        # EMA 반응속도 (높을수록 빠름)
        'MIN_WEIGHT': 0,     # 코인별 최소 비중
        'MAX_WEIGHT': 80,    # 코인별 최대 비중
        'FUND_LEVERAGE': 5,  # 펀드 전체 레버리지
        'color': '#e74c3c'
    },
    'conservative': {
        'MAX_STEP': 10,
        'ALPHA': 0.25,
        'MIN_WEIGHT': 0,
        'MAX_WEIGHT': 60,
        'FUND_LEVERAGE': 2,
        'color': '#2ecc71'
    }
}

# 코인별 최근 30일 일간 변동성 (실측치 근사)
VOLATILITY = {
    'BTC':  0.030,
    'ETH':  0.040,
    'SOL':  0.055,
    'HYPE': 0.070,
    'BNB':  0.035,
}

COINS = list(VOLATILITY.keys())
print('📋 펀드 프로파일:')
for name, p in FUND_PROFILES.items():
    print(f"  {name:12s} | MAX_STEP={p['MAX_STEP']}%p | ALPHA={p['ALPHA']} | 레버리지={p['FUND_LEVERAGE']}x")

## 2. 투표 시뮬레이션 (Kahoot 스타일)

각 참여자는 코인마다 방향 하나만 선택:  
`-2 강한 숏` / `-1 숏 늘려` / `0 유지` / `+1 롱 늘려` / `+2 강한 롱`

In [ ]:
def simulate_votes(participants, coins):
    """
    participants: [{'name': str, 'deposit': float, 'votes': {coin: int(-2~+2)}}, ...]
    반환: {coin: {'score': float, 'breakdown': dict}}
    """
    total_deposit = sum(p['deposit'] for p in participants)
    results = {}

    for coin in coins:
        weighted_score = 0
        breakdown = {-2: 0, -1: 0, 0: 0, 1: 0, 2: 0}
        for p in participants:
            vote = p['votes'].get(coin, 0)
            weight = p['deposit'] / total_deposit
            weighted_score += vote * weight
            breakdown[vote] += p['deposit']
        results[coin] = {
            'score': weighted_score / 2,  # -1 ~ +1 정규화
            'breakdown': breakdown,
            'total': total_deposit
        }
    return results


def print_vote_result(vote_result):
    print(f"{'코인':<6} {'score':>8}  방향  {'분포 (예치금 기준)':<40}")
    print('-' * 70)
    for coin, r in vote_result.items():
        score = r['score']
        direction = '🟢 롱▲' if score > 0.1 else ('🔴 숏▼' if score < -0.1 else '⚪ 유지')
        bar = ''
        labels = {-2:'강숏', -1:'숏', 0:'유지', 1:'롱', 2:'강롱'}
        for k in [-2, -1, 0, 1, 2]:
            pct = r['breakdown'][k] / r['total'] * 100
            if pct > 0:
                bar += f"{labels[k]}:{pct:.0f}% "
        print(f"{coin:<6} {score:>+8.3f}  {direction}  {bar}")


# ── 예시 투표 시나리오 ──────────────────────────────────────────
example_participants = [
    {'name': 'Alice', 'deposit': 500,
     'votes': {'BTC': 2, 'ETH': 1, 'SOL': -1, 'HYPE': 2, 'BNB': 0}},
    {'name': 'Bob',   'deposit': 300,
     'votes': {'BTC': 1, 'ETH': 0, 'SOL': 1,  'HYPE': 1, 'BNB': -1}},
    {'name': 'Carol', 'deposit': 150,
     'votes': {'BTC': 0, 'ETH': -1, 'SOL': 2, 'HYPE': -1, 'BNB': 1}},
    {'name': 'Dave',  'deposit': 50,
     'votes': {'BTC': -1, 'ETH': 2, 'SOL': 0, 'HYPE': 0, 'BNB': 2}},
]

vote_result = simulate_votes(example_participants, COINS)
print('📊 투표 결과:'); print_vote_result(vote_result)

## 3. 비중 업데이트 알고리즘 4가지 비교

In [ ]:
def algo_fixed_step(current, score, coin, profile):
    """방법 1: 고정 STEP — 단순 선형"""
    step = profile['MAX_STEP']
    return current + score * step


def algo_conviction(current, score, coin, profile):
    """방법 2: 확신도 비례 — 만장일치일수록 크게 움직임"""
    conviction = abs(score)  # 0~1
    step = profile['MAX_STEP'] * conviction
    return current + score * step


def algo_vol_adjusted(current, score, coin, profile):
    """방법 3: 변동성 연동 — 위험한 코인은 조심스럽게"""
    vol = VOLATILITY.get(coin, 0.05)
    vol_factor = 0.03 / vol  # BTC 변동성 기준으로 정규화
    conviction = abs(score)
    step = profile['MAX_STEP'] * conviction * vol_factor
    return current + score * step


def algo_combined(current, score, coin, profile):
    """방법 4: 통합 (확신도 + 변동성 + 독식방지 + EMA)"""
    vol = VOLATILITY.get(coin, 0.05)
    vol_factor = 0.03 / vol
    conviction = abs(score)
    concentration_limit = 1 - (current / 100)  # 비중 클수록 제한
    concentration_limit = max(0.1, concentration_limit)  # 최소 0.1

    raw_target = current + score * profile['MAX_STEP'] * conviction * vol_factor * concentration_limit

    # EMA 완충
    alpha = profile['ALPHA']
    return alpha * raw_target + (1 - alpha) * current


ALGORITHMS = {
    'Fixed Step':   algo_fixed_step,
    'Conviction':   algo_conviction,
    'Vol-Adjusted': algo_vol_adjusted,
    'Combined':     algo_combined,
}


def apply_algo(current_weights, vote_result, profile, algo_fn):
    """알고리즘 적용 후 합계 100% 정규화, 범위 클리핑"""
    raw = {}
    for coin in COINS:
        score = vote_result[coin]['score']
        new_w = algo_fn(current_weights[coin], score, coin, profile)
        raw[coin] = np.clip(new_w, profile['MIN_WEIGHT'], profile['MAX_WEIGHT'])

    # 합계 100% 정규화
    total = sum(abs(v) for v in raw.values())
    if total == 0:
        return {c: 0 for c in COINS}
    return {c: raw[c] / total * 100 for c in COINS}


# 현재 포트폴리오 (초기 동일 비중)
current_weights = {coin: 20.0 for coin in COINS}

print(f"{'':20s}" + "".join(f"{c:>8}" for c in COINS) + "  (합계)")
print(f"{'현재 비중':20s}" + "".join(f"{current_weights[c]:>7.1f}%" for c in COINS))
print('-' * 70)
for profile_name, profile in FUND_PROFILES.items():
    for algo_name, algo_fn in ALGORITHMS.items():
        new_w = apply_algo(current_weights, vote_result, profile, algo_fn)
        row = f"[{profile_name[:4]}] {algo_name:15s}"
        vals = "".join(f"{new_w[c]:>7.1f}%" for c in COINS)
        total = sum(new_w.values())
        print(f"{row:20s}{vals}  ({total:.0f}%)")

## 4. 롱/숏 + 레버리지 포지션 산출

In [ ]:
def weights_to_positions(weights, vote_result, tvl, profile):
    """
    비중 + 투표 방향 → 실제 노셔널 포지션 산출
    score > 0 : 롱, score < 0 : 숏, score ≈ 0 : 스킵
    """
    leverage = profile['FUND_LEVERAGE']
    positions = {}

    for coin in COINS:
        score = vote_result[coin]['score']
        weight_pct = weights[coin]

        if weight_pct < 1.0:  # 1% 미만 스킵 (수수료 낭비)
            positions[coin] = {'side': 'skip', 'notional': 0, 'weight': 0}
            continue

        side = 'long' if score >= 0 else 'short'
        notional = (weight_pct / 100) * tvl * leverage

        positions[coin] = {
            'side': side,
            'weight': weight_pct,
            'notional': notional,
            'score': score,
        }
    return positions


TVL = 100_000  # USDC

# Combined 알고리즘 + 두 프로파일로 포지션 산출
for profile_name, profile in FUND_PROFILES.items():
    new_w = apply_algo(current_weights, vote_result, profile, algo_combined)
    positions = weights_to_positions(new_w, vote_result, TVL, profile)

    print(f"\n{'='*60}")
    print(f"프로파일: {profile_name.upper()} | TVL: ${TVL:,} | 레버리지: {profile['FUND_LEVERAGE']}x")
    print(f"{'코인':<6} {'방향':<6} {'비중':>7} {'노셔널':>12} {'score':>8}")
    print('-' * 50)
    total_notional = 0
    for coin, pos in positions.items():
        if pos['side'] == 'skip':
            continue
        side_emoji = '🟢롱' if pos['side'] == 'long' else '🔴숏'
        print(f"{coin:<6} {side_emoji:<6} {pos['weight']:>6.1f}%  "
              f"${pos['notional']:>10,.0f}  {pos['score']:>+8.3f}")
        total_notional += pos['notional']
    print(f"{'합계':>14} {100:>6.0f}%  ${total_notional:>10,.0f}")

## 5. 알고리즘별 반응 비교 시각화

In [ ]:
# score 변화에 따른 각 알고리즘의 비중 변화 곡선
scores = np.linspace(-1, 1, 200)
current = 30.0  # 현재 비중 30%
coin = 'BTC'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('알고리즘별 Score → 비중 변화 반응 곡선 (현재 비중 30%)', fontsize=14, fontweight='bold')

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for ax_idx, (profile_name, profile) in enumerate(FUND_PROFILES.items()):
    ax = axes[ax_idx]
    for (algo_name, algo_fn), color in zip(ALGORITHMS.items(), colors):
        results = [algo_fn(current, s, coin, profile) for s in scores]
        ax.plot(scores, results, label=algo_name, color=color, linewidth=2)

    ax.axhline(y=current, color='gray', linestyle='--', alpha=0.5, label=f'현재 {current}%')
    ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('투표 Score (-1=강한 숏, +1=강한 롱)', fontsize=11)
    ax.set_ylabel('업데이트된 비중 (%)', fontsize=11)
    ax.set_title(f'{profile_name.upper()} 프로파일 (MAX_STEP={profile["MAX_STEP"]}%, ALPHA={profile["ALPHA"]})', fontsize=12)
    ax.legend()
    ax.set_xlim(-1, 1)
    ax.fill_betweenx([0, 100], -1, 0, alpha=0.03, color='red')
    ax.fill_betweenx([0, 100], 0, 1, alpha=0.03, color='green')

plt.tight_layout()
plt.savefig('algo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ algo_comparison.png 저장')

## 6. 다회차 투표 시뮬레이션 — 비중 변화 추적

In [ ]:
np.random.seed(42)

def random_vote_scenario(participants, coins, scenario='random'):
    """시나리오별 투표 생성"""
    for p in participants:
        for coin in coins:
            if scenario == 'random':
                p['votes'][coin] = np.random.choice([-2, -1, 0, 1, 2],
                                                     p=[ 0.1, 0.2, 0.4, 0.2, 0.1])
            elif scenario == 'bullish':
                p['votes'][coin] = np.random.choice([0, 1, 2], p=[0.2, 0.4, 0.4])
            elif scenario == 'bearish':
                p['votes'][coin] = np.random.choice([-2, -1, 0], p=[0.4, 0.4, 0.2])
    return participants


ROUNDS = 12  # 12주 (분기)
scenarios_to_test = ['random', 'bullish', 'bearish']
scenario_labels = {'random': '랜덤 투표', 'bullish': '강세장 투표', 'bearish': '약세장 투표'}

fig, axes = plt.subplots(len(FUND_PROFILES), len(scenarios_to_test),
                          figsize=(18, 10))
fig.suptitle('다회차 투표 — 포트폴리오 비중 변화 추적', fontsize=15, fontweight='bold')

coin_colors = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SOL': '#9945FF',
               'HYPE': '#00D4FF', 'BNB': '#F3BA2F'}

for p_idx, (profile_name, profile) in enumerate(FUND_PROFILES.items()):
    for s_idx, scenario in enumerate(scenarios_to_test):
        ax = axes[p_idx][s_idx]

        weights_history = {coin: [20.0] for coin in COINS}
        current_w = {coin: 20.0 for coin in COINS}

        participants = [
            {'name': 'A', 'deposit': 500, 'votes': {}},
            {'name': 'B', 'deposit': 300, 'votes': {}},
            {'name': 'C', 'deposit': 200, 'votes': {}},
        ]

        for round_i in range(ROUNDS):
            participants = random_vote_scenario(participants, COINS, scenario)
            vr = simulate_votes(participants, COINS)
            new_w = apply_algo(current_w, vr, profile, algo_combined)
            current_w = new_w
            for coin in COINS:
                weights_history[coin].append(new_w[coin])

        for coin in COINS:
            ax.plot(weights_history[coin], label=coin,
                    color=coin_colors[coin], linewidth=2, marker='o', markersize=3)

        ax.axhline(y=20, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax.set_ylim(0, 80)
        ax.set_xlabel('투표 라운드 (주)')
        ax.set_ylabel('비중 (%)')
        ax.set_title(f'{profile_name.upper()} | {scenario_labels[scenario]}')
        if p_idx == 0 and s_idx == len(scenarios_to_test) - 1:
            ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('multiround_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ multiround_weights.png 저장')

## 7. 알고리즘 선택 가이드

| 알고리즘 | 특징 | 추천 상황 |
|----------|------|----------|
| Fixed Step | 단순, 예측 가능 | 테스트/데모 |
| Conviction | 만장일치 강조 | 의견 분산 많을 때 |
| Vol-Adjusted | 리스크 균형 | 코인 변동성 차이 클 때 |
| **Combined** | **모든 요소 통합** | **실제 운용 권장** |

### 파라미터 설명 (플랫폼 문서용)

- **MAX_STEP**: 한 투표 라운드에서 비중이 변할 수 있는 최대 폭 (%). 클수록 공격적.  
- **ALPHA**: EMA 반응 속도. 0에 가까울수록 천천히 반응, 1에 가까울수록 즉각 반응.  
- **FUND_LEVERAGE**: 전체 포트폴리오 레버리지 배수.

In [ ]:
# ── 최종 출력: 투표 → 포지션 전체 플로우 요약 ──────────────────
print('=' * 65)
print('  GovernanceFund 투표 → 포지션 플로우 (Combined 알고리즘)')
print('=' * 65)

profile = FUND_PROFILES['aggressive']
new_w = apply_algo(current_weights, vote_result, profile, algo_combined)
positions = weights_to_positions(new_w, vote_result, TVL, profile)

print(f"\nTVL: ${TVL:,} USDC | 레버리지: {profile['FUND_LEVERAGE']}x")
print(f"총 노셔널: ${TVL * profile['FUND_LEVERAGE']:,} USDC\n")

print(f"{'코인':<6} {'투표score':>10} {'비중':>8} {'방향':>6} {'노셔널':>14}")
print('-' * 55)
for coin in COINS:
    pos = positions[coin]
    score = vote_result[coin]['score']
    side = '🟢 LONG' if pos['side'] == 'long' else ('🔴 SHORT' if pos['side'] == 'short' else '  SKIP')
    print(f"{coin:<6} {score:>+10.3f} {pos['weight']:>7.1f}%  {side}  ${pos['notional']:>12,.0f}")